# arg-position-back-functions — ex1: write div_back0 and div_back1 — asymmetric per-arg back fns

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `arg-position-back-functions`. Running the final beacon cell reports progress against the `Backprop: Arg-position back funcs` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)
# === manual autograd primitives — shared across all drills in this folder ===
from dataclasses import dataclass, field
from typing import Any, Callable, Optional

grad_tracking_enabled = True

@dataclass
class Recipe:
    func: Optional[Callable] = None
    args: tuple = ()
    kwargs: dict = field(default_factory=dict)
    parents: dict = field(default_factory=dict)

class MiniTensor:
    """A minimal Tensor wrapper for the ARENA-style manual-autograd drills.
    Wraps a raw `torch.Tensor` in `.array`. Carries an optional `.recipe`
    populated by wrap_forward_fn. `requires_grad` is set by the wrapper."""
    def __init__(self, array, requires_grad: bool = False, recipe=None):
        self.array = array
        self.requires_grad = requires_grad
        self.recipe = recipe
    def __repr__(self):
        return f'MiniTensor({self.array!r}, requires_grad={self.requires_grad})'

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `Backprop: Arg-position back funcs` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`arg-position-back-functions`** (exercise 1). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "arg-position-back-functions"
DD_SUBTOPIC = "Backprop: Arg-position back funcs"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## Arg-position back fns — quick refresher

Binary ops register **two** back fns — one per input position — because the gradient w.r.t. the left input and the gradient w.r.t. the right input are different functions when the op is asymmetric:

```
out = x / y
d(out)/dx = 1 / y          # 'div_back0', for arg-0 (x)
d(out)/dy = -x / y**2      # 'div_back1', for arg-1 (y)
```

Convention:
- ``f_back0(grad_out, out, x, y)`` returns ``dL/dx``
- ``f_back1(grad_out, out, x, y)`` returns ``dL/dy``

Both take **all** original args (so they can use either one), and both return a tensor with the same shape as the input they correspond to. Symmetric ops (add, multiply) still get two registrations, even if the function bodies are identical — uniform dispatch.

### Exercise 1 — write div_back0 and div_back1 — asymmetric per-arg back fns

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply the per-arg-position back-fn convention by writing div_back0 and div_back1 for out = x / y — the canonical asymmetric binary op where the two gradients are different functions.
> Keywords: arg-position, div, asymmetric, back0, back1
> ```

**KCs targeted:** `arg-position-back-functions`, `backward-fn-signature`

Implement TWO back fns for `out = x / y`, both with the same signature `(grad_out, out, x, y) -> Tensor`:

**1. `div_back0(grad_out, out, x, y)`** — gradient w.r.t. arg-0 (`x`).
   - Math: `d(x/y)/dx = 1/y`.
   - So `dL/dx = grad_out / y`.

**2. `div_back1(grad_out, out, x, y)`** — gradient w.r.t. arg-1 (`y`).
   - Math: `d(x/y)/dy = -x / y**2`.
   - So `dL/dy = grad_out * (-x / y**2)`, OR equivalently `-grad_out * out / y` (since `out = x/y` ⇒ `out/y = x/y**2`).

**Why the split.** Division is *asymmetric* — `div_back0` and `div_back1` are different functions. Compare with `add` or `multiply` (`add_back0 == add_back1`, just `grad_out`), where the two bodies happen to be identical but BOTH still get registered into BACK_FUNCS at argnum=0 AND argnum=1.

For this drill assume `x.shape == y.shape == out.shape` — no broadcasting (that's a separate atom). Inputs and outputs are plain `torch.Tensor`, no autograd, float dtype. Return tensors with the same shape as the input each back fn corresponds to.

In [ ]:
def div_back0(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    # d(x/y)/dx = 1/y, so dL/dx = grad_out / y.
    return grad_out / y


def div_back1(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    # d(x/y)/dy = -x/y^2, so dL/dy = grad_out * (-x / y^2).
    return grad_out * (-x / (y * y))


<details><summary>Solution</summary>

```python
def div_back0(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    # d(x/y)/dx = 1/y, so dL/dx = grad_out / y.
    return grad_out / y


def div_back1(grad_out: Tensor, out: Tensor, x: Tensor, y: Tensor) -> Tensor:
    # d(x/y)/dy = -x/y^2, so dL/dy = grad_out * (-x / y^2).
    return grad_out * (-x / (y * y))
```

**Why TWO functions for one op.** The argnum is part of the BACK_FUNCS lookup key: `(torch.divide, 0) -> div_back0`, `(torch.divide, 1) -> div_back1`. The reverse pass walks `recipe.parents` (a `{argnum: Tensor}` dict), and for each entry calls the matching back fn. So registration always pairs: `BACK_FUNCS.add_back_func(t.divide, 0, div_back0); BACK_FUNCS.add_back_func(t.divide, 1, div_back1)`.

**Symmetric ops still register twice.** `add_back0(g, out, x, y) = g` and `add_back1(g, out, x, y) = g` are the same function body, but you still register at both argnums. The dispatcher doesn't know whether an op is symmetric — it just looks up `(func, argnum)` and calls.

**Equivalent form for `div_back1`.** Since `out = x/y`, we have `-x/y**2 = -out/y`. Either form works; using `out` saves one multiplication and matches the 'use the cached out' pattern, at the cost of being slightly less obvious as the partial derivative.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex1'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex1',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()